# 13 — Publication Figures and Tables Generator

This notebook converts the validated experimental datasets into reproducible, publication-oriented figures and tables for the adaptive QEM manuscript.

**Principle:** figures are generated only from available measured/simulated datasets. Missing hardware results are reported as pending rather than replaced with invented values.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "Adaptive_QEM_IBM").exists() and not (ROOT / "data").exists():
    ROOT = ROOT / "Adaptive_QEM_IBM"

RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)


## 1. Load datasets

In [ ]:
def load_csv(path):
    if path.exists():
        return pd.read_csv(path)
    print("Pending:", path)
    return pd.DataFrame()

summary = load_csv(TABLES / "adaptive_vs_fixed_qem_summary.csv")
coverage = load_csv(TABLES / "adaptive_policy_coverage.csv")
overhead = load_csv(TABLES / "qem_execution_overhead_by_circuit.csv")

hardware_results = load_csv(
    ROOT / "data/hardware/extracted/ibm_sampler_v2_extracted_results.csv"
)

display(summary)
display(coverage)


## 2. Publication table — adaptive versus fixed strategies

The table reports descriptive experimental quantities. It deliberately avoids ranking language or a universal conclusion.


In [ ]:
if not summary.empty:
    pub_summary = summary.copy()

    numeric_cols = [
        c for c in pub_summary.columns
        if c not in {"comparison_strategy"}
    ]

    for c in numeric_cols:
        pub_summary[c] = pd.to_numeric(pub_summary[c], errors="coerce")

    pub_summary.to_csv(
        TABLES / "Table_Adaptive_vs_Fixed_QEM.csv",
        index=False
    )

    display(pub_summary)
else:
    print("Summary table is pending real experimental results.")


## 3. Figure 1 — Adaptive policy coverage

In [ ]:
if not coverage.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(coverage["selected_method"], coverage["fraction"])
    ax.set_xlabel("Selected QEM strategy")
    ax.set_ylabel("Fraction of benchmarks")
    ax.set_title("Adaptive QEM Policy Coverage")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(FIGURES / "Fig1_adaptive_policy_coverage.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("Coverage data pending.")


## 4. Figure 2 — Mean success probability

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(summary["comparison_strategy"], summary["mean_success"])
    ax.set_xlabel("Experimental strategy")
    ax.set_ylabel("Mean success probability")
    ax.set_title("Mean Benchmark Success Probability")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(FIGURES / "Fig2_mean_success_probability.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("Success summary pending.")


## 5. Circuit-level comparison data

In [ ]:
comparison = load_csv(
    ROOT / "data/hardware/extracted/qem_comparison_results.csv"
)

if comparison.empty:
    print("Circuit-level comparison data pending.")
else:
    display(comparison.head())


## 6. Figure 3 — Circuit-level strategy comparison

In [ ]:
if not comparison.empty and {"circuit","comparison_strategy","success_probability"}.issubset(comparison.columns):
    pivot = comparison.pivot_table(
        index="circuit",
        columns="comparison_strategy",
        values="success_probability",
        aggfunc="mean"
    )

    ax = pivot.plot(kind="bar", figsize=(12, 6))
    ax.set_xlabel("Benchmark circuit")
    ax.set_ylabel("Success probability")
    ax.set_title("Circuit-Level Success Probability by Strategy")
    ax.tick_params(axis="x", rotation=70)
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(FIGURES / "Fig3_circuit_strategy_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("Circuit-level comparison data pending.")


## 7. Figure 4 — Adaptive improvement over raw

In [ ]:
if not comparison.empty and "raw_success" in comparison.columns:
    comparison["absolute_improvement"] = (
        comparison["success_probability"] - comparison["raw_success"]
    )

    adaptive = comparison[
        comparison["comparison_strategy"] == "ADAPTIVE"
    ].copy()

    if not adaptive.empty:
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.bar(adaptive["circuit"], adaptive["absolute_improvement"])
        ax.axhline(0, linewidth=1)
        ax.set_xlabel("Benchmark circuit")
        ax.set_ylabel("Adaptive − raw success probability")
        ax.set_title("Adaptive QEM Improvement over Raw Hardware")
        ax.tick_params(axis="x", rotation=70)
        fig.tight_layout()
        fig.savefig(
            FIGURES / "Fig4_adaptive_improvement_over_raw.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)
    else:
        print("Adaptive results pending.")


## 8. Figure 5 — Reliability versus execution overhead

In [ ]:
if not overhead.empty and "execution_rows" in overhead.columns:
    oh = overhead.groupby("execution_method")["execution_rows"].mean().reset_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(oh["execution_method"], oh["execution_rows"])
    ax.set_xlabel("Execution method")
    ax.set_ylabel("Mean execution rows per circuit")
    ax.set_title("Experimental Execution Overhead")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(
        FIGURES / "Fig5_execution_overhead.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)
else:
    print("Overhead data pending.")


## 9. Figure 6 — Distribution fidelity / TVD

In [ ]:
fidelity_file = ROOT / "data/hardware/extracted/qem_distribution_metrics.csv"

if fidelity_file.exists():
    fid = pd.read_csv(fidelity_file)
    display(fid.head())

    if "distribution_fidelity" in fid.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        fid.groupby("strategy")["distribution_fidelity"].mean().plot(kind="bar", ax=ax)
        ax.set_ylabel("Mean distribution fidelity")
        ax.set_xlabel("Strategy")
        ax.set_title("Distribution Fidelity by Strategy")
        fig.tight_layout()
        fig.savefig(FIGURES / "Fig6_distribution_fidelity.png", dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)
else:
    print("Distribution-fidelity dataset pending.")


## 10. Figure 7 — Circuit complexity versus hardware error

This figure requires a merged dataset containing transpiled depth/CX count and measured hardware error. It should be interpreted as an empirical relationship for the tested backend and benchmark suite, not a universal law.


In [ ]:
complexity_file = ROOT / "data/hardware/hardware_campaign_characterization.csv"

if complexity_file.exists() and not comparison.empty:
    comp = pd.read_csv(complexity_file)
    merged = comparison.merge(comp, on="circuit", how="left")

    if "error_probability" in merged.columns and "depth" in merged.columns:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(merged["depth"], merged["error_probability"])
        ax.set_xlabel("Transpiled circuit depth")
        ax.set_ylabel("Measured error probability")
        ax.set_title("Circuit Depth versus Hardware Error")
        fig.tight_layout()
        fig.savefig(FIGURES / "Fig7_depth_vs_error.png", dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)
else:
    print("Complexity/error dataset pending.")


## 11. Figure 8 — CX count versus hardware error

In [ ]:
if complexity_file.exists() and not comparison.empty:
    comp = pd.read_csv(complexity_file)
    merged = comparison.merge(comp, on="circuit", how="left")

    if "error_probability" in merged.columns and "cx_count" in merged.columns:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(merged["cx_count"], merged["error_probability"])
        ax.set_xlabel("Transpiled CX count")
        ax.set_ylabel("Measured error probability")
        ax.set_title("CX Count versus Hardware Error")
        fig.tight_layout()
        fig.savefig(FIGURES / "Fig8_cx_vs_error.png", dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)
else:
    print("CX/error dataset pending.")


## 12. Figure 9 — Reliability/overhead trade-off

For the manuscript, this figure is particularly useful because mitigation should not be evaluated only by accuracy. A method that improves a metric but requires substantially more physical executions should be reported with that cost.


In [ ]:
if not summary.empty and not overhead.empty:
    oh = overhead.groupby("execution_method")["execution_rows"].mean().reset_index()

    trade = summary.merge(
        oh,
        left_on="comparison_strategy",
        right_on="execution_method",
        how="left"
    )

    if "mean_success" in trade.columns:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(trade["execution_rows"], trade["mean_success"])

        for _, r in trade.iterrows():
            ax.annotate(
                r["comparison_strategy"],
                (r["execution_rows"], r["mean_success"]),
                xytext=(5, 5),
                textcoords="offset points"
            )

        ax.set_xlabel("Mean execution rows per circuit")
        ax.set_ylabel("Mean success probability")
        ax.set_title("Reliability–Execution Overhead Trade-off")
        fig.tight_layout()
        fig.savefig(
            FIGURES / "Fig9_reliability_overhead_tradeoff.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)
else:
    print("Trade-off data pending.")


## 13. Publication figure index

The generated figure names are standardized for manuscript preparation.


In [ ]:
figure_index = [
    ("Fig1", "Adaptive Policy Coverage"),
    ("Fig2", "Mean Success Probability"),
    ("Fig3", "Circuit-Level Strategy Comparison"),
    ("Fig4", "Adaptive Improvement over Raw"),
    ("Fig5", "Execution Overhead"),
    ("Fig6", "Distribution Fidelity"),
    ("Fig7", "Depth versus Error"),
    ("Fig8", "CX Count versus Error"),
    ("Fig9", "Reliability–Execution Overhead Trade-off"),
]

figure_index_df = pd.DataFrame(
    figure_index,
    columns=["Figure", "Description"]
)
figure_index_df.to_csv(
    TABLES / "Publication_Figure_Index.csv",
    index=False
)
display(figure_index_df)


## 14. Publication readiness check

A figure should be considered manuscript-ready only after the underlying hardware data are complete, job IDs are preserved, calibration provenance is available, and the figure is reproducible from the stored CSV/JSON files.


In [ ]:
required_files = [
    ROOT / "data/calibration/ibm_kingston_calibration_manifest.json",
    ROOT / "data/hardware/adaptive_qem_hardware_campaign_manifest.json",
    ROOT / "data/hardware/extracted/ibm_sampler_v2_extracted_results.csv",
]

readiness = pd.DataFrame([
    {"artifact": str(f), "available": f.exists()}
    for f in required_files
])

display(readiness)
print("All required provenance artifacts available:", bool(readiness["available"].all()))
